In [ ]:
import sys
import os
from datetime import datetime
import nest_asyncio
nest_asyncio.apply()

now = datetime.now()
# The format string uses codes for Year, Month, Day, Hour (24h), Minute, and Second
formatted_datetime = now.strftime("%Y%m%d%H%M%S")

print(formatted_datetime)

devkit_path = 'nuplan-devkit'

planner_path = 'Diffusion-Planner'

print(f"Updated sys.path:\n{sys.path}\n")
os.environ["NUPLAN_EXP_ROOT"] = "/home/user_218/shaditaymor/nuplan_results"  # pick your path
os.makedirs(os.environ["NUPLAN_EXP_ROOT"], exist_ok=True)
try:
    # Import the main function
    from nuplan.planning.script.run_simulation import main as run_simulation_main

    # Define the arguments 
    script_path = 'nuplan-devkit/nuplan/planning/script/run_simulation.py'
    args = [
        script_path, 
        f"experiment_name=testing_{formatted_datetime}",
        "scenario_builder=nuplan_mini",
        "scenario_builder.map_root=/home/user_218/shaditaymor/data/data1/maps",
        "scenario_builder.data_root=/home/user_218/shaditaymor/data/data1/nuplan-v1.1/splits/mini", # Kept as absolute path
        "+simulation=closed_loop_nonreactive_agents",
        "planner=diffusion_planner",
        "planner.diffusion_planner.config.args_file=/home/user_218/shaditaymor/Project-diffusion/Diffusion-Planner/checkpoints/args.json",
        "planner.diffusion_planner.ckpt_path=/home/user_218/shaditaymor/Project-diffusion/Diffusion-Planner/checkpoints/model.pth",
        "scenario_filter.shuffle=true",
        "scenario_filter.limit_total_scenarios=2",
        "worker=sequential",
        "verbose=true",
        "hydra.searchpath=[pkg://diffusion_planner.config.scenario_filter,pkg://diffusion_planner.config,pkg://nuplan.planning.script.config.common,pkg://nuplan.planning.script.experiments]"
    ]

    # Run the simulation
    print("Backing up original sys.argv...")
    original_argv = list(sys.argv)
    
    try:
        print("Setting new sys.argv for hydra...")
        sys.argv = args
        print(f"Running simulation with args: {sys.argv}")
        
        # Call the hydra-decorated main function
        run_simulation_main()
        
        print("\nSimulation finished.")
        
    except Exception as e:
        print(f"\nAn error occurred during simulation: {e}")
        import traceback
        traceback.print_exc()
        
    finally:
        # Restore the original sys.argv so it doesn't affect other notebook cells
        print("Restoring original sys.argv...")
        sys.argv = original_argv

except ImportError as e:
    print(f"Error: Failed to import modules: {e}")
    print("Please double-check the paths set in this cell:")
    print(f"Devkit path: {devkit_path}")
    print(f"Planner path: {planner_path}")
    print("Ensure these directories exist and contain the correct packages.")




In [ ]:
import sys
import os
import glob
import nest_asyncio

nest_asyncio.apply()

# Make sure your python can import nuplan-devkit
devkit_path = 'nuplan-devkit'
if devkit_path not in sys.path:
    sys.path.insert(0, devkit_path)

from nuplan.planning.script.run_nuboard import main as run_nuboard_main

# ----------------------------
# Settings
# ----------------------------
PORT = 7007
MAX_RUNS = 20  # load last N experiments
NUBOARD_GLOB = "/home/user_218/shaditaymor/nuplan_results/exp/*/*/*/nuboard_*.nuboard"

# Find newest .nuboard files
nuboard_files = sorted(glob.glob(NUBOARD_GLOB), key=os.path.getmtime, reverse=True)[:MAX_RUNS]
if not nuboard_files:
    raise RuntimeError(f"No .nuboard files found with pattern: {NUBOARD_GLOB}")

# Hydra list formatting: simulation_path=[file1,file2,...]
simulation_path_override = "simulation_path=[" + ",".join(nuboard_files) + "]"

print("Loading nuBoard files:")
for f in nuboard_files:
    print("  -", f)

# Build argv for hydra
script_path = "/home/user_218/shaditaymor/Project-diffusion/nuplan-devkit/nuplan/planning/script/run_nuboard.py"
args = [
    script_path,
    f"port_number={PORT}",
    "worker=sequential",
    simulation_path_override,
    "scenario_builder.data_root=/home/user_218/shaditaymor/data/data1/nuplan-v1.1/splits/mini",
    "scenario_builder.map_root=/home/user_218/shaditaymor/data/data1/maps",
]

# Run
original_argv = list(sys.argv)
try:
    sys.argv = args
    print("\nStarting nuBoard with args:\n", "\n".join(sys.argv))
    run_nuboard_main()
finally:
    sys.argv = original_argv
